# Word2Vec (Skip-gram) From Scratch in NumPy

In this notebook we build Word2Vec from scratch using only NumPy.

## What is Word2Vec?

Word2Vec converts words into numbers (vectors) so that a computer can work with them.
The big idea is:

> **Words that appear in similar situations should have similar vectors.**

Example:
- `apples` and `bananas` both appear near `like` and `eat` in our sentences
- So after training, their vectors should point in a similar direction

## What is Skip-gram?

Skip-gram is the training strategy used in Word2Vec:

> Given a **center word**, predict the words around it (the context words).

Example from `"I like apples"`:
- Center = `like` → predict `I` and `apples`
- Center = `I` → predict `like`

## What we build:
| Step | What happens |
|------|--------------|
| 1 | Build a tiny corpus of sentences |
| 2 | Turn words into one-hot vectors |
| 3 | Create (center, context) training pairs |
| 4 | Build the neural network (2 weight matrices) |
| 5 | Train with forward pass + backpropagation |
| 6 | Read off the learned word embeddings |
| 7 | Walk through one training step by hand |

## Step 1 — The Corpus

We use 3 short sentences as our entire dataset.
This is small enough to trace every number by hand.

In [ ]:
import numpy as np

doc1 = "I like apples"
doc2 = "I like bananas"
doc3 = "I eat apples"

documents = [doc1, doc2, doc3]

for doc in documents:
    print(doc)

## Step 2 — Vocabulary and One-Hot Encoding

**Vocabulary:** Collect every unique word across all sentences.
Each word gets a unique index number.

**One-hot vector:** A list of zeros with exactly one `1` at the word's index position.

Example with vocab = [I, like, apples, bananas, eat]:
- `I`       = [1, 0, 0, 0, 0]
- `like`    = [0, 1, 0, 0, 0]
- `apples`  = [0, 0, 1, 0, 0]

This is how we feed words into a neural network — as numbers.

In [ ]:
# Build vocabulary (preserve order of appearance)
vocab = []
for doc in documents:
    for word in doc.split():
        if word not in vocab:
            vocab.append(word)

vocab_size = len(vocab)
word_index = {word: i for i, word in enumerate(vocab)}

print("Vocabulary:", vocab)
print("Size:", vocab_size)
print("Word to index:", word_index)

In [ ]:
def one_hot(word):
    vec = np.zeros(vocab_size, dtype=float)
    vec[word_index[word]] = 1.0
    return vec

# Show one-hot vectors for each word
print('One-hot encodings:')
for word in vocab:
    print(f'  {word:8s} -> {one_hot(word).astype(int)}')

## Step 3 — Skip-gram Training Pairs

For each word in a sentence, we look at the words next to it (within a window).
We create a pair: **(center word, neighbor word)**.

With `window_size = 1` on `"I like apples"`:

| Center | Context |
|--------|---------|
| I | like |
| like | I |
| like | apples |
| apples | like |

These pairs are the training examples.
We will teach the network: given the center word, predict the context word.

In [ ]:
window_size = 1
pairs = []

for doc in documents:
    words = doc.split()
    for i, center in enumerate(words):
        for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
            if j != i:
                pairs.append((center, words[j]))

print(f"Total training pairs: {len(pairs)}")
print("\nAll pairs:")
for p in pairs:
    print(f'  {p[0]:8s} -> {p[1]}')

## Step 4 — The Neural Network Architecture

Word2Vec is a tiny neural network with:
- **Input layer:** one-hot vector of the center word (size = vocab_size = V)
- **Hidden layer:** a small dense vector (size = embedding_dim = D)
- **Output layer:** a probability score for every word in the vocabulary (size = V)

Two weight matrices do all the work:

| Matrix | Shape | Role |
|--------|-------|------|
| W1 | V x D | Each **row** = embedding vector of a word |
| W2 | D x V | Helps decode the hidden vector into predictions |

**Key insight:** After training, W1 is what we actually want.
Each row of W1 is the learned vector for a word.

### Forward pass (making a prediction):
```
x      = one-hot vector of center word   (size V)
h      = x @ W1                          (size D)  <- embedding lookup
u      = h @ W2                          (size V)  <- scores for each word
y_hat  = softmax(u)                      (size V)  <- probabilities
```

Note: multiplying a one-hot vector by W1 just picks out one row of W1.
So the hidden layer is literally just the embedding of the center word.

In [ ]:
embedding_dim = 2   # keep small so we can visualize
V = vocab_size
D = embedding_dim

np.random.seed(0)
W1 = np.random.randn(V, D) * 0.01   # shape: (5, 2)
W2 = np.random.randn(D, V) * 0.01   # shape: (2, 5)

print("W1 shape (vocab x embedding):", W1.shape)
print("W2 shape (embedding x vocab):", W2.shape)
print("\nInitial W1 (each row = one word embedding):")
for word in vocab:
    print(f'  {word:8s} -> {W1[word_index[word]]}')

In [ ]:
def softmax(u):
    u = u - np.max(u)          # subtract max for numerical stability
    exp_u = np.exp(u)
    return exp_u / np.sum(exp_u)

def forward(center_word, W1, W2):
    x = one_hot(center_word)   # one-hot input vector
    h = x @ W1                 # hidden layer = embedding of center word
    u = h @ W2                 # output scores
    y_hat = softmax(u)         # convert scores to probabilities
    return x, h, u, y_hat

## Step 5 — Training (Forward + Backward + Update)

### Loss function
We use **cross-entropy loss**. It measures how wrong our prediction is.

We want P(context word) to be high. The loss is:
```
loss = -log( P(context word) )
```
- If P(context word) = 0.9 → loss = 0.10 (good prediction)
- If P(context word) = 0.1 → loss = 2.30 (bad prediction)

### Backpropagation (how we compute gradients)
We use the chain rule to figure out: how should we nudge each weight to reduce the loss?

The gradient of the loss with respect to the output scores is simple:
```
du = y_hat - y      (predicted probabilities minus the true one-hot target)
```

Then we propagate backwards:
```
dW2 = outer(h, du)          gradient for W2
dh  = W2 @ du               gradient flowing back to hidden layer
dW1 = outer(x, dh)          gradient for W1 (only center word's row changes)
```

### Weight update (gradient descent)
```
W1 = W1 - learning_rate * dW1
W2 = W2 - learning_rate * dW2
```

In [ ]:
def train_step(center_word, context_word, W1, W2, lr=0.1):
    # Forward pass
    x, h, u, y_hat = forward(center_word, W1, W2)
    y = one_hot(context_word)

    # Loss
    loss = -np.log(y_hat[word_index[context_word]] + 1e-12)

    # Backward pass
    du  = y_hat - y           # gradient at output
    dW2 = np.outer(h, du)     # gradient for W2
    dh  = W2 @ du             # gradient flowing to hidden
    dW1 = np.outer(x, dh)     # gradient for W1

    # Update weights
    W1 -= lr * dW1
    W2 -= lr * dW2

    return loss, W1, W2

## Step 6 — Run Training

We loop over all (center, context) pairs many times (epochs).
Each pass, the weights get slightly better at predicting context words.
We print the total loss every 50 epochs — it should go down over time.

In [ ]:
epochs = 300
lr = 0.1
loss_history = []

for epoch in range(1, epochs + 1):
    total_loss = 0
    for center, context in pairs:
        loss, W1, W2 = train_step(center, context, W1, W2, lr=lr)
        total_loss += loss
    loss_history.append(total_loss)

    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d}  |  Total Loss: {total_loss:.4f}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.grid(True)
plt.tight_layout()
plt.show()
print("Loss is going down — the model is learning.")

## Step 7 — Inspect Learned Embeddings

After training, each row of `W1` is the learned vector for that word.

We use **cosine similarity** to measure how close two word vectors are:
- Score **1.0** = vectors point in the exact same direction (very similar meaning)
- Score **0.0** = vectors are perpendicular (unrelated)
- Score **-1.0** = vectors point in opposite directions

We expect `apples` and `bananas` to be similar because they appear in the same contexts
(`I like apples`, `I like bananas`, `I eat apples`).

In [ ]:
print('Learned word embeddings (W1 rows):')
for word in vocab:
    vec = W1[word_index[word]]
    print(f'  {word:8s} -> [{vec[0]:+.4f}, {vec[1]:+.4f}]')

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

def most_similar(word, topn=4):
    v = W1[word_index[word]]
    sims = [(w, cosine_similarity(v, W1[word_index[w]])) for w in vocab if w != word]
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:topn]

print('\nMost similar words (by cosine similarity):')
for word in vocab:
    sims = most_similar(word)
    sim_str = ', '.join(f'{w}({s:.2f})' for w, s in sims)
    print(f'  {word:8s} -> {sim_str}')

In [ ]:
# Plot the 2D embeddings so we can see clusters visually
plt.figure(figsize=(6, 5))
for word in vocab:
    x_coord, y_coord = W1[word_index[word]]
    plt.scatter(x_coord, y_coord, s=100)
    plt.annotate(word, (x_coord, y_coord), textcoords='offset points', xytext=(6, 4), fontsize=12)

plt.title('Word Embeddings in 2D Space\n(words close together = similar context)')
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 8 — Manual Walkthrough: One Training Step

Let's trace through a **single training step** by hand for the pair:
- Center word: `"I"`
- Context word: `"like"`

We use **fresh random weights** so the before vs after difference is clearly visible.

We will see:
1. The probability of `like` given `I` before the update
2. The probability of `like` given `I` after the update (it should go up)
3. How the embedding of `I` changes

In [ ]:
# Use fresh weights so the change is clearly visible
np.random.seed(7)
W1_demo = np.random.randn(V, D) * 0.5
W2_demo = np.random.randn(D, V) * 0.5

center_word  = 'I'
context_word = 'like'

x = one_hot(center_word)    # one-hot for 'I'
y = one_hot(context_word)   # one-hot for 'like' (this is what we want to predict)

print('Input x (one-hot for I):   ', x.astype(int))
print('Target y (one-hot for like):', y.astype(int))

In [ ]:
# --- Forward pass ---
h     = x @ W1_demo        # hidden = embedding of 'I'
u     = h @ W2_demo        # output scores for all words
y_hat = softmax(u)         # convert to probabilities

print('Hidden vector h (embedding of I):', h.round(4))
print('Output scores u:                 ', u.round(4))
print()
print('Probabilities y_hat:')
for word, prob in zip(vocab, y_hat):
    marker = '  <-- target' if word == context_word else ''
    print(f'  P({word:8s}) = {prob:.4f}{marker}')

loss_before = -np.log(y_hat[word_index[context_word]] + 1e-12)
print(f'\nLoss before update: {loss_before:.4f}')

In [ ]:
# --- Backward pass ---
du  = y_hat - y            # how wrong was each output
dW2 = np.outer(h, du)      # gradient for W2
dh  = W2_demo @ du         # gradient flowing back to hidden
dW1 = np.outer(x, dh)      # gradient for W1

print('Gradient at output (du = y_hat - y):')
for word, g in zip(vocab, du):
    print(f'  {word:8s}: {g:+.4f}')

print(f'\nGradient for embedding of I (dW1 row):', dW1[word_index['I']].round(4))

In [ ]:
lr = 0.5

print('Embedding of I  BEFORE update:', W1_demo[word_index['I']].round(4))

# Update weights
W1_demo -= lr * dW1
W2_demo -= lr * dW2

print('Embedding of I  AFTER  update:', W1_demo[word_index['I']].round(4))

# Check that P('like') increased
h_new     = x @ W1_demo
u_new     = h_new @ W2_demo
y_hat_new = softmax(u_new)

loss_after = -np.log(y_hat_new[word_index[context_word]] + 1e-12)

print()
print(f'P(like | I) BEFORE: {y_hat[word_index["like"]]:.4f}')
print(f'P(like | I) AFTER:  {y_hat_new[word_index["like"]]:.4f}  (should be higher)')
print(f'Loss       BEFORE:  {loss_before:.4f}')
print(f'Loss       AFTER:   {loss_after:.4f}  (should be lower)')

## Summary

Here is everything we did in one place:

| Concept | Explanation |
|---------|-------------|
| **One-hot vector** | Represents a word as a list of zeros with a single 1 |
| **W1** (V x D) | The embedding table — each row is a word's learned vector |
| **W2** (D x V) | Decoder matrix — helps predict context words |
| **Forward pass** | x → h = x@W1 → u = h@W2 → y_hat = softmax(u) |
| **Loss** | -log(P(true context word)) — lower is better |
| **Backward pass** | du = y_hat - y, then gradients for W1 and W2 |
| **Update** | W = W - learning_rate * gradient |
| **Embedding** | After training, W1 rows are the word vectors we use |

**Why does this work?**

Every time the network sees `(I, like)` as a pair, it nudges the vector for `I`
to become a bit better at predicting `like`.
It also sees `(I, eat)` as a pair, so `I` also becomes better at predicting `eat`.
Words like `apples` and `bananas` are both paired with `like` and `eat`,
so their vectors end up pointing in similar directions.
That is the entire learning mechanism.